# 06. Memory Optimization & Type Precision (5+ Years Interview Guide)
Exhaustive revision guide to typecasting with .astype(), float32 downcasting (50% RAM savings), overflow testing, and iinfo/finfo on raw_transactions.csv.

### Key 5-Year Interview Concepts Covered:
- **Typecasting (`.astype`)**: Converting dtypes across int8, int16, int32, int64, float32, and float64.
- **Precision Management**: Downcasting float64 to float32 cutting memory footprint by exactly 50%.
- **Integer Overflow Bounds**: Understanding two's complement boundary rollover in fixed-width integers.
- **Inspecting Numeric Limits**: Dedicated cell for `np.iinfo()` and `np.finfo()`.

This interactive revision guide loads and operates directly on `data/raw_transactions.csv` using dedicated cells per method.

In [ ]:
# Setup imports & dataset loading from raw_transactions.csv
import numpy as np
import pandas as pd
import sys
import time
import os

# Load raw transactions and extract aligned NumPy numeric arrays
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
raw_df = pd.read_csv(csv_path)
clean_raw = raw_df.dropna(subset=['transaction_amount', 'is_fraud', 'account_age_months']).reset_index(drop=True)
amounts = clean_raw['transaction_amount'].to_numpy(dtype=np.float64)
fraud_flags = clean_raw['is_fraud'].to_numpy(dtype=np.int8)
account_ages = clean_raw['account_age_months'].to_numpy(dtype=np.float32)

print(f"NumPy Version: {np.__version__}")
print(f"Loaded from {csv_path} ({len(amounts)} clean aligned rows):")
print(f"- amounts array: shape {amounts.shape}, dtype {amounts.dtype}")
print(f"- fraud_flags array: shape {fraud_flags.shape}, dtype {fraud_flags.dtype}")
print(f"- account_ages array: shape {account_ages.shape}, dtype {account_ages.dtype}")

### Typecasting with `.astype()`
**Explanation**: Casts transaction amounts from float64 (8 bytes) to float32 (4 bytes).

**Syntax**: `amounts.astype(np.float32)`

In [ ]:
amounts_f32 = amounts.astype(np.float32)
print('Original Dtype:', amounts.dtype, '-> New Dtype:', amounts_f32.dtype)

### Precision Management: 50% RAM Reduction
**Explanation**: Measures memory savings from downcasting clean transaction amounts.

**Syntax**: `amounts.nbytes` vs `amounts_f32.nbytes`

In [ ]:
print(f'float64 Memory: {amounts.nbytes / 1024:.2f} KB')
print(f'float32 Memory: {amounts_f32.nbytes / 1024:.2f} KB (exactly 50% RAM saved)')

### Integer Overflow Boundary Testing
**Explanation**: Demonstrates rollover when casting high account ages into `int8`.

**Syntax**: `np.array([127], dtype=np.int8) + 1`

In [ ]:
overflow_demo = np.array([125, 126, 127], dtype=np.int8)
overflow_demo += 1
print('int8 Two\'s Complement Overflow Result (127 + 1 -> -128):', overflow_demo)

### Inspecting Numeric Limits with `iinfo` & `finfo`
**Explanation**: Inspects machine limits for `int8`, `int16`, and `float32`.

**Syntax**: `np.iinfo(np.int8)` / `np.finfo(np.float32)`

In [ ]:
print('int8 Range:', np.iinfo(np.int8).min, 'to', np.iinfo(np.int8).max)
print('int16 Range:', np.iinfo(np.int16).min, 'to', np.iinfo(np.int16).max)
print('float32 Machine Epsilon:', np.finfo(np.float32).eps)

## Section: Senior Fintech Interview Scenarios (5+ Years Experience)

### Q1: Automated Safe Downcasting on Transaction Vectors
**Explanation**: Write a function to downcast transaction amounts and account ages to the smallest safe dtype without overflow.

**Syntax**: `np.iinfo` checks

In [ ]:
def safe_downcast(arr):
    if np.issubdtype(arr.dtype, np.floating):
        return arr.astype(np.float32)
    elif np.issubdtype(arr.dtype, np.integer):
        for dt in [np.int8, np.int16, np.int32]:
            if arr.min() >= np.iinfo(dt).min and arr.max() <= np.iinfo(dt).max:
                return arr.astype(dt)
    return arr

print('Downcasted Fraud Flags Dtype:', safe_downcast(fraud_flags).dtype)
print('Downcasted Amounts Dtype:', safe_downcast(amounts).dtype)